# 03 · 手刻 ReAct 迴圈

整條軌道的**靈魂課**。上一課只跑一輪；真實任務常是多步的——「現在到下午三點還有幾小時」要先**查時間**再**算差**。需要一個迴圈。

**ReAct** 把每一步拆成三段，反覆進行直到收斂：

```
Thought（想）→ Action（做）→ Observation（看）→ … → Final Answer
```

## 1. 載入模型與工具

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

In [ ]:
from datetime import datetime, timezone, timedelta


def get_current_time() -> str:
    """回傳台北現在時間（模型沒有時鐘，這是它拿不到的真實世界資訊）。"""
    tz = timezone(timedelta(hours=8))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")


def calculator(expression: str) -> str:
    """安全地算一個算式。只允許數字與 + - * / ( ) . %。"""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression or ""):
        return "錯誤：只接受數字與 + - * / ( ) . %"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"錯誤：{e}"

In [ ]:
def parse_action(text):
    """從模型輸出抓出 (工具名, 參數dict)；抓不到回 (None, None)。"""
    m_name = re.search(r"Action:\s*([A-Za-z_]\w*)", text)
    if not m_name:
        return None, None
    args = {}
    m_args = re.search(r"Action Input:\s*(\{.*\})", text, re.DOTALL)
    if m_args:
        try:
            args = json.loads(m_args.group(1))
        except Exception:
            args = {}
    return m_name.group(1), args

## 2. ReAct 的 system prompt

約定 Thought / Action / Final Answer 的格式。

In [ ]:
REACT_SYSTEM = """你是一個會使用工具、一步步推理的助手。可用工具：

- get_current_time()：回傳現在台北時間，無參數。
- calculator(expression)：計算算式字串，例如 {"expression": "15-9"}。

請嚴格依這個格式，一次只走一步：

Thought: 你的推理
Action: 工具名稱
Action Input: JSON 參數

我會以 Observation 回覆工具結果，然後你再繼續下一個 Thought。
當資訊足夠時，改用這個格式收尾：

Thought: 最後推理
Final Answer: 給使用者的答案"""

## 3. 手刻迴圈

兩個工程細節：**停止條件**（看到 `Final Answer:` 就收工）與 **`max_steps` 護欄**（防止模型鬼打牆無限迴圈）。並把模型輸出在 `Observation` 處截斷，不讓它自己幻想工具結果。

In [ ]:
TOOLS = {"get_current_time": get_current_time, "calculator": calculator}


def run_react(question, max_steps=6, verbose=True):
    scratchpad = ""
    for step in range(1, max_steps + 1):
        messages = [{"role": "system", "content": REACT_SYSTEM},
                    {"role": "user", "content": f"問題：{question}\n\n{scratchpad}"}]
        reply = chat(messages, max_new_tokens=256)
        reply = reply.split("Observation")[0].strip()   # 別讓模型自己幻想觀察
        if verbose:
            print(f"--- 第 {step} 步 ---\n{reply}")
        if "Final Answer:" in reply:
            return reply.split("Final Answer:")[-1].strip()
        name, args = parse_action(reply)
        if name is None:
            obs = "（沒解析到 Action，請用正確格式，或給 Final Answer）"
        elif name not in TOOLS:
            obs = f"沒有名為 {name} 的工具。"
        else:
            obs = TOOLS[name](**args)
        if verbose:
            print(f"Observation: {obs}\n")
        scratchpad += f"{reply}\nObservation: {obs}\n"
    return "（達到最大步數仍未完成）"

## 4. 丟一個需要連續兩步的問題

In [ ]:
answer = run_react("現在台北時間的『小時』數字，加上 100 是多少？")
print("\n========\n最終答案：", answer)

## 小結

- **ReAct = Thought → Action → Observation** 反覆，把只會吐字的 LLM 變成**會多步推理、自己用工具**的 agent。
- 手刻迴圈必備：**停止條件**（`Final Answer:`）與 **`max_steps` 護欄**。
- 在 `Observation` 處截斷模型輸出，避免它幻想結果。

下一課：把兩支寫死的工具，升級成**一整箱可擴充、會路由**的工具系統。